#[5] Multi Agent

In [ ]:
!pip install -qU langgraph langchain langchain-openai

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv(dotenv_path = "/content/.env", override=True)

print(".env 내 OPENAI_API_KEY가 환경변수에 할당됐습니다:", os.environ["OPENAI_API_KEY"][:5]+"*****")

#1 . Command with Goto

##1-1. 사용자“yes”,“no” 답변

###① Model 정의

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5-nano")

###② State 정의

In [ ]:
from langchain.messages import AnyMessage
from typing_extensions import TypedDict, Annotated
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]    # add_messages 리듀서(reducer)로 누적
    selected_menu: str # 선택된 메뉴 저장

###③ Node 정의

In [ ]:
# [Node 1] 메뉴 추천
def recommender_node(state: ChatState):
    # 현재 상태(이전에 추천했던 메뉴)를 확인
    last_menu = state.get("selected_menu")

    # 만약 이전에 A코스를 추천했다면 -> 거절당하고 다시 온 것이므로 B코스 추천
    if last_menu == "A 코스 (20만원)":
        print("\n🍣 [Recommender] 그렇다면. 실속형 'B 코스 (8만원)'을 추천합니다.")
        return {"selected_menu": "B 코스 (8만원)"}

    # 기본(최초) 추천
    else:
        print("\n🍣 [Recommender] 쉐프 추천: 'A 코스 (20만원)'")
        return {"selected_menu": "A 코스 (20만원)"}

In [ ]:
from langgraph.types import interrupt, Command

# [Node 2] 승인/거절 및 경로 변경 (Router 역할 겸임)
def human_approval_node(state: ChatState):
    menu = state["selected_menu"]
    print(f"***메뉴 '{menu}'의 사용자 확정 필요***\n")

    # 1. 멈추고 물어봄 (Interrupt)
    approved = interrupt(f"{menu} 예약을 진행하시겠습니까? (yes/no)")

    # 2. 답변에 따라 즉시 순간이동 (Command with goto)
    if approved == "yes":
        print(f"사용자가 {menu}를 확정하였습니다\n")
        return Command(
            update={"messages" : [f"사용자가 {menu} 를 확정하였습니다"]},
            goto="booking"
        )
    else:
        print(f"사용자가 {menu}를 거절하였습니다. 다른 메뉴 추천이 필요합니다\n")
        return Command(
            update={"messages" : [f"사용자가 거절했습니다. 다른 메뉴를 추천합니다"]},
            goto="recommender"
        )


In [ ]:
# [Node 3] 예약 확정 (승인 시 이동할 곳)
def booking_node(state: ChatState):
    menu = state["selected_menu"]
    print(f"\n✅ [Booking] '{menu}' 예약이 확정되었습니다! (문자 발송 완료)")
    return {"messages": [f"'{menu}' 예약 완료"]}

###④ 그래프 생성

In [ ]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(ChatState)

workflow.add_node("recommender", recommender_node)
workflow.add_node("approval", human_approval_node)
workflow.add_node("booking", booking_node)

In [ ]:
workflow.add_edge(START, "recommender")
workflow.add_edge("recommender", "approval")
workflow.add_edge("approval", END)

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

In [ ]:
app = workflow.compile(checkpointer=checkpointer)

In [ ]:
app

###⑤ 실행

In [ ]:
config = {"configurable": {"thread_id": "goto-demo"}}

In [ ]:
from langchain.messages import HumanMessage

app.invoke({"messages": [HumanMessage(content="예약해줘")]}, config=config)

In [ ]:
snapshot = app.get_state(config)
print(f"⚠️ 대기 중: {snapshot.tasks[0].interrupts[0].value}")

In [ ]:
app.invoke(Command(resume="no"), config=config)

In [ ]:
snapshot = app.get_state(config)
print(f"{snapshot.tasks[0].interrupts[0].value}")

In [ ]:
result = app.invoke(Command(resume="yes"), config=config)

In [ ]:
result['messages'][-1].content

##1-2. 사용자 자연어 답변

###① Model 정의

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

# 1. 원하는 출력 구조 정의 (Pydantic Schema)
class UserDecision(BaseModel):
    reason: str = Field(
        description="사용자의 답변에서 예약 의사가 확실한지 분석한 이유를 간단히 적으세요."
    )
    response: Literal["yes", "no"] = Field(
        description="최종 분류 결과 (명확한 예약 확정만 'yes', 거절/가격 불만/대안 요구/망설임은 모두 'no')"
    )

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# 2. 사용자 답변 분류 모델 체인 정의 (structured output, ChatPromptTemplate)
router_llm = model.with_structured_output(UserDecision)

prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 사용자의 답변을 분석하여 예약 의사 여부를 정확히 분류하는 분류기입니다.

분류 규칙:
1. 'yes': 사용자가 추천된 메뉴를 명확히 수락하고 예약 진행을 원할 때만 부여합니다.
2. 'no': 사용자가 거절하거나, 가격이 비싸다고 하거나, 다른 메뉴 추천을 요청하거나, 망설이는 등 명확한 예약 의사가 보이지 않을 때 부여합니다.

반드시 사용자의 속뜻을 파악하여 규칙에 맞춰 정밀하게 분류하세요.
"""
    ),
    ("user", "사용자 답변: {user_input}")
])

classifier_chain = prompt | router_llm

###② State 정의 - 기정의된 ChatState 사용

###③ Node 정의

In [ ]:
from langgraph.types import interrupt, Command

# [Node 2] 승인/거절 및 경로 변경 (Router 역할 겸임)
def human_approval_node(state: ChatState):
    menu = state["selected_menu"]
    print(f"***메뉴 '{menu}'의 사용자 확정 필요***\n")

    # 1. 멈추고 물어봄 (Interrupt)
    user_input = interrupt(f"{menu} 예약을 진행하시겠습니까?")
    print(f"\n--- [Router] 답변 분석 중: '{user_input}' ---")

    # LLM이 문맥을 읽고 카테고리를 결정함 (Rule-based가 아님!)
    decision = classifier_chain.invoke({"user_input": user_input})
    approved = decision.response
    print(f"*** 사용자가 답변 ===> {approved}  ***\n")

    # 2. 답변에 따라 즉시 순간이동 (Command with goto)
    if approved == "yes":
        print(f"사용자가 {menu}를 확정하였습니다\n")
        return Command(
            update={"messages" : [f"사용자가 {menu} 를 확정하였습니다"]},
            goto="booking"
        )
    else:
        print(f"사용자가 {menu}를 거절하였습니다. 다른 메뉴 추천이 필요합니다\n")
        return Command(
            update={"messages" : [f"사용자가 거절했습니다. 다른 메뉴를 추천합니다"]},
            goto="recommender"
        )

###④ 그래프 생성

In [ ]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(ChatState)

workflow.add_node("recommender", recommender_node)
workflow.add_node("approval", human_approval_node)
workflow.add_node("booking", booking_node)

In [ ]:
workflow.add_edge(START, "recommender")
workflow.add_edge("recommender", "approval")
workflow.add_edge("approval", END)

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

In [ ]:
app = workflow.compile(checkpointer=checkpointer)

In [ ]:
app

###⑤ 실행

In [ ]:
config = {"configurable": {"thread_id": "goto-demo-01"}}

In [ ]:
from langchain.messages import HumanMessage

app.invoke({"messages": [HumanMessage(content="예약해줘")]}, config=config)

In [ ]:
snapshot = app.get_state(config)
print(f"⚠️ 대기 중: {snapshot.tasks[0].interrupts[0].value}")

In [ ]:
app.invoke(Command(resume="너무 비싸요. 다른 메뉴 추천해줘요"), config=config)

In [ ]:
snapshot = app.get_state(config)
print(f"{snapshot.tasks[0].interrupts[0].value}")

In [ ]:
app.invoke(Command(resume="네, 좋네요"), config=config)

#2 . Supervisor - 중앙집중형
[공식 문서](https://docs.langchain.com/oss/python/langchain/multi-agent)

##2-1. Model 정의

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5-nano")

##2-2. State 정의

In [ ]:
from langchain.messages import AnyMessage, SystemMessage, HumanMessage
from typing_extensions import TypedDict, Annotated
from langgraph.graph.message import add_messages

class TeamState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

##2-3. Supervisor Node

###① NextStep 결정 모델 정의

In [ ]:
# 감독관이 내릴 수 있는 결정 목록
class RouteResponse(BaseModel):
    next_step: Literal["researcher", "writer",  "FINISH"] = Field(
        description="다음 작업을 수행할 팀원을 선택하거나, 모든 작업이 완료되었으면 FINISH를 선택"
    )
    reason: str = Field(description="선택한 이유")

In [ ]:
supervisor_llm = model.with_structured_output(RouteResponse)

###② Node 정의

In [ ]:
def supervisor_node(state: TeamState):
    print("\n👔 [Supervisor] 다음 작업자 결정 중...")

    # 1. 현재까지의 대화 내용을 보고 판단
    system_prompt = """
    당신은 블로그 작성 팀의 관리자입니다.
    1. 사용자 요청이 들어오면 'researcher'에게 조사를 시키세요.
    2. 조사 결과가 나오면 'writer'에게 글 작성을 시키세요.
    3. 글 작성이 완료되면 'FINISH'를 선언하세요.
    """

    # LLM에게 판단 요청
    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    decision = supervisor_llm.invoke(messages)

    print(f"\n👔 [Supervisor] 다음 작업자 결정 -> {decision.next_step}")

    # 2. 판단 결과에 따라 라우팅 (Command Goto)
    if decision.next_step == "FINISH":
        return Command(goto=END)
    else:
        return Command(goto=decision.next_step)

##2-4. Worker Node

###① Node 정의

In [ ]:
# [팀원 1] 검색 담당 (Researcher)
def researcher_node(state: TeamState):
    print("\n🔎 [Researcher] LLM이 자료를 조사/생성 중입니다...")

    # 시스템 프롬프트 부여
    system_msg = SystemMessage(content="""
    당신은 IT 트렌드 전문 리서처입니다.
    사용자의 요청 주제에 대해 핵심 트렌드, 기술 용어, 통계 등을 포함하여
    전문적인 조사 보고서를 3줄 요약 형태로 작성하세요.
    """)

    # 이전 대화 내용 + 내 역할(System)을 합쳐서 LLM에게 전달
    # state["messages"] 안에 사용자의 요청이 들어있음
    messages = [system_msg] + state["messages"]

    # LLM 실행
    response = model.invoke(messages)

    print(f"\n🔎 [Researcher] LLM이 자료를 조사/생성 완료 -> {response.content}")

    # 결과 반환 (name을 지정해야 감독관이 누가 말했는지 앎)
    return Command(update={"messages": [HumanMessage(content=response.content, name="researcher")]},
                   goto="supervisor")


In [ ]:
# [팀원 2] 집필 담당 (Writer)
def writer_node(state: TeamState):
    print("\n✍️ [Writer] LLM이 블로그 포스팅을 작성 중입니다...")

    system_msg = SystemMessage(content="""
    당신은 테크 블로그 전문 작가입니다.
    위의 대화 내역에 있는 'Researcher'의 조사 결과를 바탕으로
    매력적인 블로그 포스팅(제목+본문)을 작성하세요.
    이모지를 적절히 사용하여 가독성을 높이세요.
    """)

    # 공유된 대화 기록(Researcher의 결과 포함)을 다 같이 넘겨줌
    messages = [system_msg] + state["messages"]
    response = model.invoke(messages)

    print(f"\n✍️ [Writer] LLM이 블로그 포스팅을 작성 완료 -> {response.content}")

    return Command(update={"messages" : [HumanMessage(content=response.content, name="writer")]},
                   goto = "supervisor")


##2-5. 그래프 생성

In [ ]:
workflow = StateGraph(TeamState)

workflow.add_node("supervisor", supervisor_node)
workflow.add_node("researcher", researcher_node)
workflow.add_node("writer", writer_node)

workflow.add_edge(START, "supervisor")

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

In [ ]:
app = workflow.compile(checkpointer=checkpointer)

In [ ]:
app

##2-6. 실행

In [ ]:
config = {"configurable": {"thread_id": "team-blog-1"}}

In [ ]:
app.invoke(
    {"messages": [HumanMessage(content="AI 최신 트렌드에 대해 블로그 글 써줘")]},
    config=config
)

새로운 작업자 '검수자'를 추가한다면 어떻게 구현하면 될까?

#3 . Subgraph

##3-1. State 공유

###① Model 정의

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5-nano")

###② State 정의

In [ ]:
from langchain.messages import AnyMessage
from typing_extensions import TypedDict, Annotated
from langgraph.graph.message import add_messages

class ReservationState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    menu: str
    price: int
    status: str
    chef_name: str
    cooking_time: str

###③ Sub Graph (주방 시스템)

####③-1. Node 정의

In [ ]:
# 주방만의 별도 워크플로우를 정의
def check_ingredients(state: ReservationState):
    print(f"\n   🍳 [주방/재료팀] '{state['menu']}' 재료 재고 확인 중...")
    # (로직 생략: 무조건 재료가 있다고 가정)
    return {"status": "ok"}

In [ ]:
def assign_chef(state: ReservationState):
    menu = state['menu']
    if "오마카세" in menu:
        chef = "Master Jiro"
        time = "40분"
    else:
        chef = "Chef Kim"
        time = "20분"

    print(f"   👨‍🍳 [주방/인사팀] {chef} 쉐프 배정 완료 (예상 소요: {time})")
    return {"chef_name": chef, "cooking_time": time, "status": "cooking"}

####③-2. 그래프 생성

In [ ]:
kitchen_builder = StateGraph(ReservationState)
kitchen_builder.add_node("check_stock", check_ingredients)
kitchen_builder.add_node("assign_chef", assign_chef)

kitchen_builder.add_edge(START, "check_stock")
kitchen_builder.add_edge("check_stock", "assign_chef")
kitchen_builder.add_edge("assign_chef", END)

In [ ]:
# 주방 그래프 컴파일 (이 자체가 하나의 노드처럼 쓰임)
kitchen_subgraph = kitchen_builder.compile()

In [ ]:
kitchen_subgraph

###④ Parent Graph (메인 홀 시스템)

####④-1. Node 정의

In [ ]:
# [부서 1] 메뉴 추천
def menu_recommender(state: ReservationState):
    last_msg = state["messages"][-1].content if state["messages"] else ""

    if "비싸" in last_msg or "부담" in last_msg:
        proposal = "B코스 (런치 스페셜)"
        price = 80000
        msg = f"🍣 부담 없는 '{proposal} ({price:,}원)'은 어떠신가요?"
    else:
        proposal = "A코스 (쉐프 오마카세)"
        price = 150000
        msg = f"🍣 시그니처 '{proposal} ({price:,}원)'를 추천합니다."

    return {"menu": proposal, "price": price, "messages": [msg]}

####④-2. 사용자의도분석모델정의

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class UserIntent(BaseModel):
    action: Literal["confirm", "change", "cancel"] = Field(description="확정, 변경, 취소 판단")

In [ ]:
intent_analyzer = model.with_structured_output(UserIntent)

####④-3. Node 정의

In [ ]:
# [부서 2] Agentic Router
def customer_confirm_node(state: ReservationState):
    menu = state['menu']
    price = state['price']

    user_input = interrupt(f"'{menu}' ({price:,}원)으로 하시겠습니까?")

    # LLM 의도 분석
    analysis = intent_analyzer.invoke(f"제안: {menu}, 답변: {user_input}")
    print(f"🧠 [AI Router]: 의도='{analysis.action}'")

    if analysis.action == "confirm":
        # 여기서 'kitchen'으로 보내면, 연결된 Subgraph가 실행됨
        return Command(
            update={"messages" : [HumanMessage(content=user_input)]},
            goto="kitchen"
        )
    elif analysis.action == "change":
        return Command(
            update={"messages" : [HumanMessage(content=user_input)]},
            goto="recommender"
        )
    else:
        return Command(
            update={"messages" : [HumanMessage(content=user_input)]},
            goto="cancel"
        )

In [ ]:
# [부서 3] 취소 처리
def cancel_node(state: ReservationState):
    print(f"\n👋 [카운터]: 예약 종료.")
    return {"status": "cancelled"}

####④-4. 그래프 생성

In [ ]:
parent_builder = StateGraph(ReservationState)

parent_builder.add_node("recommender", menu_recommender)
parent_builder.add_node("confirm", customer_confirm_node)
parent_builder.add_node("cancel", cancel_node)

In [ ]:
# 컴파일된 Subgraph를 '노드'로 등록
parent_builder.add_node("kitchen", kitchen_subgraph)

In [ ]:
# 엣지 등록
parent_builder.add_edge(START, "recommender")
parent_builder.add_edge("recommender", "confirm")
parent_builder.add_edge("kitchen", END)


In [ ]:
checkpointer = InMemorySaver()

In [ ]:
app = parent_builder.compile(checkpointer=checkpointer)

In [ ]:
app

###⑤ 실행

In [ ]:
config = {"configurable": {"thread_id": "subgraph-demo-1"}}

In [ ]:
app.invoke({"messages": []}, config=config)

In [ ]:
user_response = "좋아요, 그걸로 주세요."
result = app.invoke(Command(resume=user_response), config=config)

In [ ]:
result

In [ ]:
print(f"메뉴: {result['menu']}")
print(f"상태: {result['status']}")
print(f"담당 쉐프: {result.get('chef_name')}")

In [ ]:
config = {"configurable": {"thread_id": "subgraph-demo-2"}}

In [ ]:
app.invoke({"messages": []}, config=config)

In [ ]:
user_response = "아니요.. 너무 비싸요."
result = app.invoke(Command(resume=user_response), config=config)

In [ ]:
result

In [ ]:
user_response = "그렇게 할게요."
final_result = app.invoke(Command(resume=user_response), config=config)

In [ ]:
final_result

In [ ]:
print(f"메뉴: {final_result['menu']}")
print(f"상태: {final_result['status']}")
print(f"담당 쉐프: {final_result.get('chef_name')} (Subgraph에서 생성된 정보!)")

##3-2. State 분리

###① Sub Graph

####①-1. State 정의

In [ ]:
from typing_extensions import TypedDict

class DeliveryState(TypedDict):
    address : str
    package_info : str
    delivery_status : str


####①-2. Node 정의

In [ ]:
def pickup_package(state: DeliveryState):
    print(f"\n   🛵 [배달업체] 픽업 완료! (물품: {state['package_info']})")
    return {"delivery_status": "picked_up"}

In [ ]:
def deliver_to_customer(state: DeliveryState):
    print(f"   Hit [배달업체] {state['address']}로 배송 중... 띵동! 배달 완료.")
    return {"delivery_status": "delivered"}

####①-3. 그래프 생성

In [ ]:
delivery_builder = StateGraph(DeliveryState)

delivery_builder.add_node("pickup", pickup_package)
delivery_builder.add_node("deliver", deliver_to_customer)

delivery_builder.add_edge(START, "pickup")
delivery_builder.add_edge("pickup", "deliver")
delivery_builder.add_edge("deliver", END)

In [ ]:
# 하위 그래프 컴파일
delivery_graph = delivery_builder.compile()

In [ ]:
delivery_graph

###② Parent Graph

####②-1. State 정의


In [ ]:
# Parent State
class RestaurantState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    menu: str
    customer_address: str
    final_status: str

####②-2. Node 정의

In [ ]:
def food_manager(state: RestaurantState):
    print(f"\n👨‍🍳 [주방] '{state['menu']}' 조리 완료! 포장했습니다.")
    return {"messages": ["조리 완료"]}

In [ ]:
# 외부 그래프를 호출하는 '브릿지(Bridge)' 노드
def delivery_manager(state: RestaurantState):
    print("\n📞 [매니저] 배달 업체 호출 중...")

    # 1. 내 공책(Parent)의 정보를 -> 외부 업체 양식(Child)으로 변환 (Mapping)
    delivery_input = {
        "address": state["customer_address"],
        "package_info": state["menu"]
    }

    # 2. 함수 안에서 그래프를 실행 (API 호출하듯이)
    delivery_result = delivery_graph.invoke(delivery_input)

    # 3. 결과를 받아서 내 공책에 기록
    final_status = delivery_result["delivery_status"]
    return {"final_status" : final_status}


####②-3. 그래프 생성

In [ ]:
parent_builder = StateGraph(RestaurantState)

parent_builder.add_node("food_manager", food_manager)
parent_builder.add_node("delivery_manager", delivery_manager)

parent_builder.add_edge(START, "food_manager")
parent_builder.add_edge("food_manager", "delivery_manager")
parent_builder.add_edge("delivery_manager", END)

In [ ]:
app = parent_builder.compile()

In [ ]:
app

###③ 실행

In [ ]:
inputs = {
    "menu": "특선 초밥 세트",
    "customer_address": "서울시 강남구 역삼동 123",
    "messages": []
}

In [ ]:
result = app.invoke(inputs)

In [ ]:
result

#4 . Handoff - Network형
[LangGraph 공식 문서](https://docs.langchain.com/oss/python/langchain/multi-agent/handoffs)

[OpenAI Handoff](https://openai.github.io/openai-agents-python/handoffs/)

##4-1. Model 정의

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5-nano")

##4-2. Tool 정의

[tool runtime](https://reference.langchain.com/python/langgraph/agents/?h=toolruntime#langgraph.prebuilt.tool_node.InjectedStore)

In [ ]:
from langgraph.types import Command
from langchain.messages import ToolMessage
from langchain.tools import tool, ToolRuntime

@tool
def transfer_to_tech_support(runtime: ToolRuntime):
    """기술적인 문제(컴퓨터, 코드, 에러 등)를 해결해야 할 때 이 도구를 사용하여 기술팀으로 연결하세요."""

    tool_call_id = runtime.tool_call_id
    print(f"*** 기술지원팀(Tech Support)으로 전달합니다. tool_call_id = {tool_call_id} \n")

    return Command(
        update={"messages":
         [ToolMessage(
             content="기술지원팀(Tech Support)으로 전달합니다",
             tool_call_id=tool_call_id,
             name="transfer_to_tech_support")
         ]},
        goto="tech_support"
    )


In [ ]:
@tool
def transfer_to_billing(runtime: ToolRuntime):
    """환불, 결제, 금액 관련 문제를 해결해야 할 때 이 도구를 사용하여 환불팀으로 연결하세요."""

    tool_call_id = runtime.tool_call_id
    print(f"*** 빌링팀(Billing)으로 전달합니다. tool_call_id = {tool_call_id} \n")

    return Command(
        update={"messages":
         [ToolMessage(
             content="빌링팀(Billing)으로 전달합니다",
             tool_call_id=tool_call_id,
             name="transfer_to_billing")
         ]},
        goto="billing"
    )


In [ ]:
@tool
def transfer_to_reception(runtime: ToolRuntime):
    """일반적인 문의거나, 인사가 끝났으면 안내 데스크로 다시 연결하세요."""

    tool_call_id = runtime.tool_call_id

    print(f"*** 안내데스크(Reception)로 전달합니다. tool_call_id = {tool_call_id} \n")

    return Command(
        update={"messages":
         [ToolMessage(
             content="안내데스크(Reception)로 전달합니다",
             tool_call_id=tool_call_id,
             name="transfer_to_reception")
         ]},
        goto="receptionist"
    )


##4-3. State 정의

In [ ]:
from langchain.messages import AnyMessage, SystemMessage, HumanMessage
from typing_extensions import TypedDict, Annotated
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
      messages: Annotated[list[AnyMessage], add_messages]

##4-4. Node 정의

In [ ]:
# [상담원 1] 안내 데스크 (시작점)
def receptionist_node(state: AgentState):
    print("\n💁 [Reception] 안녕하세요.")

    system_prompt = """
    당신은 고객센터의 친절한 안내 데스크입니다.
    1. 고객의 말을 듣고 인사하세요.
    2. '기술 문제'라면 tech_support 도구를 쓰세요.
    3. '돈/환불 문제'라면 billing 도구를 쓰세요.
    4. 그 외 잡담은 직접 답변하세요.
    """

    # 안내 데스크는 기술팀과 환불팀으로 보낼 수 있음
    tools = [transfer_to_tech_support, transfer_to_billing]
    model_with_tools = model.bind_tools(tools)

    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    response = model_with_tools.invoke(messages)

    return {"messages": [response]}

In [ ]:
# [상담원 2] 기술 지원팀
def tech_node(state: AgentState):
    print("\n🛠️ [Tech Support] 기술적인 문제를 해결해드립니다.")

    system_prompt = """
    당신은 베테랑 기술 지원 엔지니어입니다.
    기술 문제 (컴퓨터 고장, 파이썬 코드 에러 등) 를 해결해줍니다.
    해결이 끝나면 reception으로 넘기세요.
    만약 고객이 '환불'을 요구하면 즉시 billing 도구를 써서 넘기세요.
    """

    # 기술팀은 환불팀이나 안내 데스크로 보낼 수 있음
    tools = [transfer_to_billing, transfer_to_reception]
    model_with_tools = model.bind_tools(tools)

    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    response = model_with_tools.invoke(messages)

    return {"messages": [response]}

In [ ]:
# [상담원 3] 환불팀
def billing_node(state: AgentState):
    print("\n💰 [Billing] 결제 및 환불 담당입니다.")

    system_prompt = """
    당신은 환불 담당자입니다.
    고객의 환불 요청을 처리해줍니다.
    해결이 끝나면 reception으로 넘기세요.
    기술적인 질문을 하면 tech_support로 넘기세요.
    """

    # 환불팀은 기술팀이나 안내 데스크로 보낼 수 있음
    tools = [transfer_to_tech_support, transfer_to_reception]
    model_with_tools = model.bind_tools(tools)

    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    response = model_with_tools.invoke(messages)

    return {"messages": [response]}

In [ ]:
from langgraph.prebuilt import ToolNode

# 도구를 실행할 노드 (ToolNode)
# 에이전트가 도구를 호출하면 -> Tool Node로 가서 -> 함수 실행 -> Command 반환 -> 다른 노드로 점프
all_tools = [transfer_to_tech_support, transfer_to_billing, transfer_to_reception]
tool_node = ToolNode(all_tools)

In [ ]:
tool_node

##4-5. 그래프 생성

In [ ]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(AgentState)

# 시작은 안내 데스크
workflow.add_edge(START, "receptionist")
workflow.add_node("receptionist", receptionist_node)
workflow.add_node("tech_support", tech_node)
workflow.add_node("billing", billing_node)

workflow.add_node("tools", tool_node)

In [ ]:
from langgraph.prebuilt import tools_condition

# 모든 노드는 tools 또는 END 노드 중 하나 (조건부 엣지)
workflow.add_conditional_edges("receptionist", tools_condition, ["tools", END])
workflow.add_conditional_edges("tech_support", tools_condition, ["tools", END])
workflow.add_conditional_edges("billing", tools_condition, ["tools", END])

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

In [ ]:
app = workflow.compile(checkpointer=checkpointer)

In [ ]:
app

##4-6. 실행

In [ ]:
config = {"configurable": {"thread_id": "handoff-demo-2"}}

In [ ]:
app.invoke(
    {"messages": [HumanMessage(content="제 컴퓨터 화면이 안 나와요. 고장난 것 같아요.")]},
    config=config
)

In [ ]:
app.invoke(
    {"messages": [HumanMessage(content="아직 안 되는데, 그냥 환불하고 싶어요.")]},
    config=config
)

In [ ]:
app.invoke(
    {"messages": [HumanMessage(content="회원 탈퇴는 어떻게 하나요?")]},
    config=config
)

##4-7. 실행 (gradio)

In [ ]:
import gradio as gr

config = {"configurable": {"thread_id": "handoff-demo-0"}}

# 1. Gradio 인터페이스 함수 정의
def respond(message, history):
    """
    message: 사용자의 질문
    history: 이전 대화 기록 (Gradio ChatInterface에서 자동 관리)
    """
    result = app.invoke(
        {"messages": [HumanMessage(content=message)]},
        config=config
    )

    # LangGraph의 최종 메시지 내용을 추출하여 반환
    return result["messages"][-1].content

# 2. Gradio 웹 화면 실행
demo = gr.ChatInterface(
    fn=respond,
    title="🤖 고객 대응 멀티 에이전트 서비스 🤖",
    description="일반 안내, 기술 문의, 환불 문의를 담당합니다"
)

if __name__ == "__main__":
    # share=True로 설정하면 외부에서 접속 가능한 임시 링크가 생성됩니다.
    demo.launch(share=True, debug=True)